In [186]:
import json
import pandas as pd
import nltk
import random
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
nltk.download("punkt")
nltk.download("punkt_tab")
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
with open("Gemini.json", "r") as file:
    data = json.load(file)
    gemini = pd.DataFrame(data)
with open("Claude.json", "r") as file:
    data = json.load(file)
    claude = pd.DataFrame(data)
with open("ChatGPT.json", "r") as file:
     data = json.load(file)
     chatGPT = pd.DataFrame(data)
gemini = gemini.rename(columns = {0:"LLM_Input", 1:"LLM_Output"})
claude = claude.rename(columns = {0:"LLM_Input", 1: "LLM_Output"})
chatGPT = chatGPT.rename(columns = {0:"LLM_Input", 1: "LLM_Output"})
gemini = gemini.head(500)
claude = claude.head(500)
chatGPT = chatGPT.head(500)
gemini["LLM_Name"] = "Gemini"
claude["LLM_Name"] = "Claude"
chatGPT["LLM_Name"] = "ChatGPT"
for i in range(0,500):
    gemini.loc[i, "LLM_Input"] = gemini["LLM_Input"][i]["content"]
    gemini.loc[i, "LLM_Output"] = gemini["LLM_Output"][i]["content"]
    claude.loc[i, "LLM_Input"] = claude["LLM_Input"][i]["content"]
    claude.loc[i, "LLM_Output"] = claude["LLM_Output"][i]["content"]
    chatGPT.loc[i, "LLM_Input"] = chatGPT["LLM_Input"][i]["content"]
    chatGPT.loc[i, "LLM_Output"] = chatGPT["LLM_Output"][i]["content"]
combined = pd.concat([gemini,chatGPT,claude], ignore_index = True)
train_X,temp_X,train_Y,temp_Y = train_test_split(combined["LLM_Output"],combined["LLM_Name"],test_size = 0.2)
test_X,val_X,test_Y,val_Y = train_test_split(temp_X, temp_Y, test_size = 0.5)
'''
combined["Input_Output"] = combined["LLM_Input"] + " " + combined["LLM_Output"]
unique_prompts = combined["LLM_Input"].unique()
train_prompts, temp_prompts = train_test_split(unique_prompts, test_size=0.2)
test_prompts, val_prompts = train_test_split(temp_prompts, test_size=0.5)
train_data = combined[combined["LLM_Input"].isin(train_prompts)]
test_data = combined[combined["LLM_Input"].isin(test_prompts)]
val_data = combined[combined["LLM_Input"].isin(val_prompts)]
train_X = train_data["Input_Output"]
train_Y = train_data["LLM_Name"]
test_X = test_data["Input_Output"]
test_Y = test_data["LLM_Name"]
val_X = val_data["Input_Output"]
val_Y = val_data["LLM_Name"]
'''
train_X = train_X.reset_index(drop = True)
tokensList = []
for i in range(0,len(train_X)):
    tokens = word_tokenize(train_X[i])
    tokensList.append(tokens)
tokenDic = {}
tokenIds = []
id_count = 1
for i in range(0,len(tokensList)):
    for j in range(0,len(tokensList[i])):
     if(tokensList[i][j] not in tokenDic):
             tokenDic[tokensList[i][j]] = id_count
             id_count +=1
max_length = 0
for i in range(0,len(tokensList)):
    current_tokens = []
    for j in range(0,len(tokensList[i])):
       current_tokens.append(tokenDic[tokensList[i][j]])
    tokenIds.append(current_tokens)
count = 0
low = 0
middle = 0
high = 0
for i in range(0,len(tokenIds)):
   count+= len(tokenIds[i])
average = count/len(tokenIds)
for i in range(0,len(tokenIds)):
    if len(tokenIds[i]) <= 500:
        low += 1
    elif len(tokenIds[i]) >= 500 and len(tokenIds[i]) <= 800:
        middle +=1
    else:
        high += 1
max_length = 800
for i in range(0,len(tokenIds)):
     if(len(tokenIds[i]) < max_length):
         for j in range(len(tokenIds[i]), max_length):
            tokenIds[i].append(0)  
     elif(len(tokenIds[i]) > max_length):
         for j in range(len(tokenIds[i]), max_length, -1):
             tokenIds[i].pop()
     else:
         pass
Output_dic = {"Claude" : 0, "Gemini" : 1, "ChatGPT" : 2}
outputs = []
train_Y = train_Y.reset_index(drop = True)
for i in range(0,len(train_Y)):
    outputs.append(Output_dic[train_Y[i]])
x = torch.tensor(tokenIds)
y = torch.tensor(outputs)
dataset = TensorDataset(x,y)
dataLoader = DataLoader(dataset, batch_size = 40)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embeddings = nn.Embedding(num_embeddings = len(tokenDic)+2,embedding_dim = 100, padding_idx = 0)
        self.conv = nn.Conv1d(in_channels=100, out_channels=256, kernel_size=5)
        self.relu = nn.ReLU()
        self.linear = nn.Linear(256, 3)
    def forward(self, x):
        embedded = self.embeddings(x)
        embedded = embedded.permute(0,2,1)
        convolved = self.conv(embedded)
        activated = self.relu(convolved)
        pooled = torch.max(activated, dim=2).values
        scores = self.linear(pooled)
        return scores
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for i in range(4):
 total_average = 0
 for batchX, batchY in dataLoader:
    optimizer.zero_grad()
    scores = model(batchX)
    loss = criterion(scores, batchY)
    total_average += loss.item()
    loss.backward()
    optimizer.step()
 print(total_average/30)
val_X = val_X.reset_index(drop = True)
valTokens = []
for i in range(0,len(val_X)):
    tokens = word_tokenize(val_X[i])
    valTokens.append(tokens)
valIds = []
UNK_ID = len(tokenDic) + 1
for i in range(0,len(valTokens)):
    current_tokens = []
    for j in range(0,len(valTokens[i])):
        if valTokens[i][j] in tokenDic :
            current_tokens.append(tokenDic[valTokens[i][j]])
        else:
            current_tokens.append(UNK_ID)
    valIds.append(current_tokens)
for i in range(0,len(valIds)):
     if(len(valIds[i]) < max_length):
         for j in range(len(valIds[i]), max_length):
           valIds[i].append(0)  
     elif(len(valIds[i]) > max_length):
         for j in range(len(valIds[i]), max_length, -1):
             valIds[i].pop()
     else:
         pass
valX = torch.tensor(valIds)
val_outputs = []
val_Y = val_Y.reset_index(drop = True)
for i in range(0,len(val_Y)):
    val_outputs.append(Output_dic[val_Y[i]])
valY = torch.tensor(val_outputs) 
valScores = model(valX)
predictions = torch.argmax(valScores, dim=1)
correct = 0
for i in range(0,len(val_outputs)):
    if(val_outputs[i] == predictions[i]):
        correct+=1
#print(correct)
test_X = test_X.reset_index(drop = True)
testTokens = []
for i in range(0,len(test_X)):
    tokens = word_tokenize(test_X[i])
    testTokens.append(tokens)
testIds = []
for i in range(0,len(testTokens)):
    current_tokens = []
    for j in range(0,len(testTokens[i])):
        if testTokens[i][j] in tokenDic :
            current_tokens.append(tokenDic[testTokens[i][j]])
        else:
            current_tokens.append(UNK_ID)
    testIds.append(current_tokens)
for i in range(0,len(testIds)):
     if(len(testIds[i]) < max_length):
         for j in range(len(testIds[i]), max_length):
           testIds[i].append(0)  
     elif(len(testIds[i]) > max_length):
         for j in range(len(testIds[i]), max_length, -1):
             testIds[i].pop()
     else:
         pass
testX = torch.tensor(testIds)
test_outputs = []
test_Y = test_Y.reset_index(drop = True)
for i in range(0,len(test_Y)):
    test_outputs.append(Output_dic[test_Y[i]])
testY = torch.tensor(test_outputs) 
testScores = model(testX)
test_predictions = torch.argmax(testScores, dim=1)
correct = 0
for i in range(0,len(test_outputs)):
    if(test_outputs[i] == test_predictions[i]):
        correct+=1
print("CNN: ", correct/150 * 100)
class LSTM(nn.Module):
     def __init__(self):
        super().__init__()
        self.embeddings = nn.Embedding(num_embeddings = len(tokenDic)+2,embedding_dim = 100, padding_idx = 0)
        self.lstm = nn.LSTM(input_size=100, hidden_size=128, batch_first=True)
        self.linear = nn.Linear(128, 3)
     def forward(self, x):
        embedded = self.embeddings(x)
        output, (h, c) = self.lstm(embedded)
        final_h = h[0]
        scores = self.linear(final_h)
        return scores
new_model = LSTM()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(new_model.parameters(), lr=0.001)
for i in range(4):
 total_average = 0
 for batchX, batchY in dataLoader:
    optimizer.zero_grad()
    scores = new_model(batchX)
    loss = criterion(scores, batchY)
    total_average += loss.item()
    loss.backward()
    optimizer.step()
 #print(total_average/30)
valScores = new_model(valX)
predictions = torch.argmax(valScores, dim=1)
correct = 0
for i in range(0,len(val_outputs)):
    if(val_outputs[i] == predictions[i]):
        correct+=1
print(correct)
testScores = new_model(testX)
test_predictions = torch.argmax(testScores, dim=1)
correct = 0
for i in range(0,len(test_outputs)):
    if(test_outputs[i] == test_predictions[i]):
        correct+=1
print(correct/150 * 100)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rishi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\rishi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


0.7932973355054855
0.32137325902779895
0.18189880500237146
0.10452838813265165
CNN:  84.0
49
41.333333333333336


In [182]:
formatting = "-*#"
average_gpt = 0
chatGPT_unique = 0
chatGPT_all = 0
chatGPT_list  = []
chatGPT_formatCount = 0
for i in range(0,500):
 average_gpt += len(word_tokenize(chatGPT["LLM_Output"][i]))
 for i in word_tokenize(chatGPT["LLM_Output"][i]):
     if not i in chatGPT_list:
         chatGPT_unique +=1
         chatGPT_list.append(i)
     if i in formatting:
        chatGPT_formatCount+=1 
     chatGPT_all +=1
print(average_gpt/500)
print(chatGPT_unique/chatGPT_all)
print(chatGPT_formatCount)
print(chatGPT_formatCount/chatGPT_all)

519.572
0.09328447260437436
23005
0.08855365570123101


In [183]:
formatting = "-*#"
average_claude = 0
claude_unique = 0
claude_all = 0
claude_list  = []
claude_formatCount = 0
for i in range(0,500):
 average_claude += len(word_tokenize(claude["LLM_Output"][i]))
 for i  in word_tokenize(claude["LLM_Output"][i]):
     if not i in claude_list:
         claude_unique +=1
         claude_list.append(i)
     if i in formatting:
         claude_formatCount+=1
     claude_all +=1
print(average_claude/500)
print(claude_unique/claude_all)
print(claude_formatCount)
print(claude_formatCount/claude_all)

385.152
0.11192983549351944
12181
0.0632529494848787


In [185]:
formatting = "-*#"
average_gemini = 0
gemini_unique = 0
gemini_all = 0
gemini_list  = []
gemini_formatCount = 0
for i in range(0,500):
  average_gemini += len(word_tokenize(gemini["LLM_Output"][i]))
  for i  in word_tokenize(gemini["LLM_Output"][i]):
     if not i in gemini_list:
       gemini_unique+=1
       gemini_list.append(i)
     if i in formatting:
        gemini_formatCount+=1 
     gemini_all+=1
print(average_gemini/500)
print(gemini_unique/gemini_all)
print(gemini_formatCount)
print(gemini_formatCount/gemini_all)

520.324
0.0934802161729999
26194
0.10068342033040951
